# 02 — Data Validation and Leakage-Safe Splitting

This notebook consumes the manifests produced by `01_dataset_exploration.ipynb` and creates reproducible, case-level train/validation/test manifests. It removes exact duplicate judgments, prevents training data from overlapping any evaluation set, preserves IN-Ext's two expert references, and exports both CSV and JSONL manifests for the preprocessing stage.

### Split policy

- **IN-Abs official train:** cleaned, deduplicated, then split 90%/10% into training and validation.
- **IN-Abs official test:** kept as the Indian abstractive test set.
- **IN-Ext:** held out completely as the expert evaluation set; A1 and A2 remain two references for the same 50 cases.
- **UK-Abs official train:** cleaned and split 90%/10%; included in active training only when `TRAINING_SCOPE='multijurisdiction'`.
- **UK-Abs official test:** held out; training copies of its exact judgment hashes are removed.

## 1. Configuration

In [2]:
from pathlib import Path
import json
import os

PROJECT_ROOT = Path(os.getenv(
    "LEGALMIND_PROJECT_ROOT",
    "/data2/user_data/sg57092c/LLM_finetune",
)).expanduser()

EDA_DIR = PROJECT_ROOT / "artifacts" / "eda"
SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
VALIDATION_REPORT_DIR = PROJECT_ROOT / "artifacts" / "data_validation"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_REPORT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
VALIDATION_FRACTION = 0.10
TRAINING_SCOPE = "india"  # choices: 'india', 'multijurisdiction'

if TRAINING_SCOPE not in {"india", "multijurisdiction"}:
    raise ValueError("TRAINING_SCOPE must be 'india' or 'multijurisdiction'")

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"EDA_DIR       : {EDA_DIR}")
print(f"SPLIT_DIR     : {SPLIT_DIR}")
print(f"TRAINING_SCOPE: {TRAINING_SCOPE}")

PROJECT_ROOT  : /data2/user_data/sg57092c/LLM_finetune
EDA_DIR       : /data2/user_data/sg57092c/LLM_finetune/artifacts/eda
SPLIT_DIR     : /data2/user_data/sg57092c/LLM_finetune/data/splits
TRAINING_SCOPE: india


## 2. Load and validate EDA manifests

In [3]:
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

case_manifest_path = EDA_DIR / "case_manifest.csv"
pair_manifest_path = EDA_DIR / "full_summary_pair_manifest.csv"

for required_path in (case_manifest_path, pair_manifest_path):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required EDA artifact not found: {required_path}. "
            "Run 01_dataset_exploration.ipynb first."
        )

case_df = pd.read_csv(case_manifest_path)
pair_df = pd.read_csv(pair_manifest_path)

required_case_columns = {
    "case_id", "dataset", "split", "judgment_member",
    "judgment_hash", "summary_count", "judgment_words"
}
required_pair_columns = {
    "case_id", "dataset", "split", "author", "judgment_member",
    "summary_member", "judgment_hash", "summary_hash"
}

missing_case_columns = required_case_columns - set(case_df.columns)
missing_pair_columns = required_pair_columns - set(pair_df.columns)
if missing_case_columns or missing_pair_columns:
    raise ValueError({
        "missing_case_columns": sorted(missing_case_columns),
        "missing_pair_columns": sorted(missing_pair_columns),
    })

# CSV converts a missing author into NaN. Normalize it before rebuilding IDs.
pair_df["author"] = pair_df["author"].fillna("reference").astype(str)
pair_df["author"] = pair_df["author"].replace({"nan": "reference", "None": "reference"})
pair_df["pair_id"] = (
    pair_df["dataset"].astype(str) + ":"
    + pair_df["split"].astype(str) + ":"
    + pair_df["case_id"].astype(str) + ":"
    + pair_df["author"]
)

print(f"Loaded {len(case_df):,} case rows and {len(pair_df):,} full-summary pairs.")
display(case_df.groupby(["dataset", "split"]).size().rename("cases").to_frame())
display(pair_df.groupby(["dataset", "split", "author"]).size().rename("pairs").to_frame())

Loaded 7,973 case rows and 8,023 full-summary pairs.


cases
dataset split             
IN-Abs  test           100
        train         7030
IN-Ext  expert_eval     50
UK-Abs  test           100
        train          693

pairs
dataset split       author          
IN-Abs  test        reference    100
        train       reference   7030
IN-Ext  expert_eval A1            50
                    A2            50
UK-Abs  test        reference    100
        train       reference    693

## 3. Validate row-level integrity

These checks stop the pipeline when a required identifier, member path, or hash is missing.

In [4]:
critical_case_columns = ["case_id", "dataset", "split", "judgment_member", "judgment_hash"]
critical_pair_columns = [
    "case_id", "dataset", "split", "judgment_member",
    "summary_member", "judgment_hash", "summary_hash"
]

case_nulls = case_df[critical_case_columns].isna().sum()
pair_nulls = pair_df[critical_pair_columns].isna().sum()
empty_summary_cases = int(case_df["summary_count"].fillna(0).eq(0).sum())
duplicate_pair_paths = int(pair_df.duplicated(["dataset", "judgment_member", "summary_member"]).sum())

integrity_report = pd.DataFrame({
    "check": [
        "Null values in critical case fields",
        "Null values in critical pair fields",
        "Cases without full summaries",
        "Duplicate judgment-summary path pairs",
    ],
    "value": [int(case_nulls.sum()), int(pair_nulls.sum()), empty_summary_cases, duplicate_pair_paths],
})
display(integrity_report)

if case_nulls.sum() or pair_nulls.sum() or empty_summary_cases or duplicate_pair_paths:
    raise ValueError("Critical data-integrity checks failed. Inspect integrity_report before splitting.")

,check,value
0,Null values in critical case fields,0
1,Null values in critical pair fields,0
2,Cases without full summaries,0
3,Duplicate judgment-summary path pairs,0


## 4. Build clean evaluation partitions

Exact duplicates inside an evaluation partition are reduced to one canonical case. Training records matching any evaluation hash are removed later.

In [5]:
decision_rows = []

def canonicalize_cases(df, partition_name):
    ordered = df.sort_values(["judgment_hash", "case_id", "judgment_member"]).copy()
    duplicate_mask = ordered.duplicated("judgment_hash", keep="first")
    for row in ordered.loc[duplicate_mask].itertuples(index=False):
        decision_rows.append({
            "dataset": row.dataset,
            "case_id": row.case_id,
            "judgment_hash": row.judgment_hash,
            "original_split": row.split,
            "decision": "removed",
            "reason": f"exact duplicate inside {partition_name}",
        })
    return ordered.loc[~duplicate_mask].copy()

in_abs_test_cases = canonicalize_cases(
    case_df[(case_df["dataset"] == "IN-Abs") & (case_df["split"] == "test")],
    "IN-Abs test",
)
in_ext_expert_cases = canonicalize_cases(
    case_df[(case_df["dataset"] == "IN-Ext") & (case_df["split"] == "expert_eval")],
    "IN-Ext expert evaluation",
)
uk_abs_test_cases = canonicalize_cases(
    case_df[(case_df["dataset"] == "UK-Abs") & (case_df["split"] == "test")],
    "UK-Abs test",
)

evaluation_cases = pd.concat(
    [in_abs_test_cases, in_ext_expert_cases, uk_abs_test_cases],
    ignore_index=True,
)
evaluation_hashes = set(evaluation_cases["judgment_hash"])

display(pd.DataFrame({
    "evaluation_partition": ["IN-Abs test", "IN-Ext expert", "UK-Abs test"],
    "clean_cases": [len(in_abs_test_cases), len(in_ext_expert_cases), len(uk_abs_test_cases)],
}))

,evaluation_partition,clean_cases
0,IN-Abs test,100
1,IN-Ext expert,50
2,UK-Abs test,99


## 5. Remove training leakage and exact duplicates

In [6]:
raw_training_pool = case_df[
    ((case_df["dataset"] == "IN-Abs") & (case_df["split"] == "train"))
    | ((case_df["dataset"] == "UK-Abs") & (case_df["split"] == "train"))
].copy()

leakage_mask = raw_training_pool["judgment_hash"].isin(evaluation_hashes)
leakage_rows = raw_training_pool.loc[leakage_mask].copy()
for row in leakage_rows.itertuples(index=False):
    decision_rows.append({
        "dataset": row.dataset,
        "case_id": row.case_id,
        "judgment_hash": row.judgment_hash,
        "original_split": row.split,
        "decision": "removed",
        "reason": "judgment hash overlaps a held-out evaluation partition",
    })

non_leaking_pool = raw_training_pool.loc[~leakage_mask].copy()
dataset_priority = {"IN-Abs": 0, "UK-Abs": 1}
non_leaking_pool["_dataset_priority"] = non_leaking_pool["dataset"].map(dataset_priority).fillna(99)
non_leaking_pool = non_leaking_pool.sort_values(
    ["judgment_hash", "_dataset_priority", "case_id", "judgment_member"]
)
duplicate_training_mask = non_leaking_pool.duplicated("judgment_hash", keep="first")
duplicate_training_rows = non_leaking_pool.loc[duplicate_training_mask].copy()
for row in duplicate_training_rows.itertuples(index=False):
    decision_rows.append({
        "dataset": row.dataset,
        "case_id": row.case_id,
        "judgment_hash": row.judgment_hash,
        "original_split": row.split,
        "decision": "removed",
        "reason": "exact duplicate inside candidate training pool",
    })

clean_training_pool = (
    non_leaking_pool.loc[~duplicate_training_mask]
    .drop(columns="_dataset_priority")
    .reset_index(drop=True)
)

cleaning_summary = pd.DataFrame({
    "stage": [
        "Raw official training cases",
        "Removed for evaluation leakage",
        "Removed as exact training duplicates",
        "Clean candidate training cases",
    ],
    "cases": [
        len(raw_training_pool), len(leakage_rows),
        len(duplicate_training_rows), len(clean_training_pool),
    ],
})
display(cleaning_summary)
display(leakage_rows.groupby("dataset").size().rename("removed_for_leakage").to_frame())

,stage,cases
0,Raw official training cases,7723
1,Removed for evaluation leakage,11
2,Removed as exact training duplicates,76
3,Clean candidate training cases,7636


,removed_for_leakage
dataset,
UK-Abs,11


## 6. Reproducible case-level train/validation split

Each jurisdiction is split independently so the validation distribution is not dominated by IN-Abs.

In [7]:
def split_train_validation(df, validation_fraction=0.10, random_seed=42):
    if df.empty:
        return df.copy(), df.copy()
    shuffled = df.sample(frac=1.0, random_state=random_seed).reset_index(drop=True)
    validation_size = max(1, int(round(len(shuffled) * validation_fraction)))
    validation = shuffled.iloc[:validation_size].copy()
    training = shuffled.iloc[validation_size:].copy()
    return training, validation

india_pool = clean_training_pool[clean_training_pool["dataset"] == "IN-Abs"].copy()
uk_pool = clean_training_pool[clean_training_pool["dataset"] == "UK-Abs"].copy()

india_train_cases, india_validation_cases = split_train_validation(
    india_pool, VALIDATION_FRACTION, RANDOM_SEED
)
uk_train_cases, uk_validation_cases = split_train_validation(
    uk_pool, VALIDATION_FRACTION, RANDOM_SEED
)

if TRAINING_SCOPE == "india":
    active_train_cases = india_train_cases.copy()
    active_validation_cases = india_validation_cases.copy()
else:
    active_train_cases = pd.concat([india_train_cases, uk_train_cases], ignore_index=True)
    active_validation_cases = pd.concat([india_validation_cases, uk_validation_cases], ignore_index=True)

split_counts = pd.DataFrame({
    "partition": [
        "India train", "India validation", "IN-Abs test", "IN-Ext expert test",
        "UK train", "UK validation", "UK-Abs test",
        "Active train", "Active validation",
    ],
    "cases": [
        len(india_train_cases), len(india_validation_cases), len(in_abs_test_cases),
        len(in_ext_expert_cases), len(uk_train_cases), len(uk_validation_cases),
        len(uk_abs_test_cases), len(active_train_cases), len(active_validation_cases),
    ],
})
display(split_counts)

,partition,cases
0,India train,6303
1,India validation,700
2,IN-Abs test,100
3,IN-Ext expert test,50
4,UK train,570
5,UK validation,63
6,UK-Abs test,99
7,Active train,6303
8,Active validation,700


## 7. Map clean cases back to judgment–summary pairs

In [8]:
def pairs_for_cases(cases, assigned_split):
    selected = cases[["dataset", "judgment_member"]].drop_duplicates()
    result = pair_df.merge(
        selected, on=["dataset", "judgment_member"], how="inner", validate="many_to_one"
    ).copy()
    result["original_split"] = result["split"]
    result["assigned_split"] = assigned_split
    result["jurisdiction"] = result["dataset"].map({
        "IN-Abs": "India", "IN-Ext": "India", "UK-Abs": "United Kingdom"
    })
    result["pair_id"] = (
        result["dataset"].astype(str) + ":"
        + result["case_id"].astype(str) + ":"
        + result["author"].astype(str)
    )
    return result.sort_values(["dataset", "case_id", "author"]).reset_index(drop=True)

active_train_pairs = pairs_for_cases(active_train_cases, "train")
active_validation_pairs = pairs_for_cases(active_validation_cases, "validation")
india_train_pairs = pairs_for_cases(india_train_cases, "train")
india_validation_pairs = pairs_for_cases(india_validation_cases, "validation")
uk_train_pairs = pairs_for_cases(uk_train_cases, "train")
uk_validation_pairs = pairs_for_cases(uk_validation_cases, "validation")
in_abs_test_pairs = pairs_for_cases(in_abs_test_cases, "test_in_abs")
in_ext_expert_pairs = pairs_for_cases(in_ext_expert_cases, "test_in_ext_expert")
uk_abs_test_pairs = pairs_for_cases(uk_abs_test_cases, "test_uk_abs")

pair_counts = pd.DataFrame({
    "partition": [
        "Active train", "Active validation", "IN-Abs test",
        "IN-Ext expert references", "UK-Abs test",
    ],
    "pairs": [
        len(active_train_pairs), len(active_validation_pairs), len(in_abs_test_pairs),
        len(in_ext_expert_pairs), len(uk_abs_test_pairs),
    ],
    "unique_cases": [
        active_train_pairs["judgment_hash"].nunique(),
        active_validation_pairs["judgment_hash"].nunique(),
        in_abs_test_pairs["judgment_hash"].nunique(),
        in_ext_expert_pairs["judgment_hash"].nunique(),
        uk_abs_test_pairs["judgment_hash"].nunique(),
    ],
})
display(pair_counts)
print("IN-Ext intentionally has two reference pairs per unique case.")

,partition,pairs,unique_cases
0,Active train,6303,6303
1,Active validation,700,700
2,IN-Abs test,100,100
3,IN-Ext expert references,100,50
4,UK-Abs test,99,99


IN-Ext intentionally has two reference pairs per unique case.


## 8. Assert zero leakage between final partitions

In [9]:
partitions = {
    "active_train": active_train_cases,
    "active_validation": active_validation_cases,
    "test_in_abs": in_abs_test_cases,
    "test_in_ext_expert": in_ext_expert_cases,
    "test_uk_abs": uk_abs_test_cases,
}

leakage_checks = []
partition_names = list(partitions)
for i, left_name in enumerate(partition_names):
    left_hashes = set(partitions[left_name]["judgment_hash"])
    for right_name in partition_names[i + 1:]:
        right_hashes = set(partitions[right_name]["judgment_hash"])
        overlap = left_hashes & right_hashes
        leakage_checks.append({
            "left_partition": left_name,
            "right_partition": right_name,
            "overlapping_judgment_hashes": len(overlap),
        })

final_leakage_report = pd.DataFrame(leakage_checks)
display(final_leakage_report)
if final_leakage_report["overlapping_judgment_hashes"].sum() != 0:
    raise AssertionError("Final partitions still contain exact-judgment leakage.")
print("PASS: all active train, validation and evaluation partitions are hash-disjoint.")

,left_partition,right_partition,overlapping_judgment_hashes
0,active_train,active_validation,0
1,active_train,test_in_abs,0
2,active_train,test_in_ext_expert,0
3,active_train,test_uk_abs,0
4,active_validation,test_in_abs,0
5,active_validation,test_in_ext_expert,0
6,active_validation,test_uk_abs,0
7,test_in_abs,test_in_ext_expert,0
8,test_in_abs,test_uk_abs,0
9,test_in_ext_expert,test_uk_abs,0


PASS: all active train, validation and evaluation partitions are hash-disjoint.


## 9. Distribution checks

In [10]:
distribution_rows = []
for partition_name, frame in {
    "train": active_train_pairs,
    "validation": active_validation_pairs,
    "test_in_abs": in_abs_test_pairs,
    "test_in_ext_expert": in_ext_expert_pairs,
    "test_uk_abs": uk_abs_test_pairs,
}.items():
    if frame.empty:
        continue
    distribution_rows.append({
        "partition": partition_name,
        "pairs": len(frame),
        "unique_cases": frame["judgment_hash"].nunique(),
        "median_judgment_words": float(frame["judgment_words"].median()),
        "p95_judgment_words": float(frame["judgment_words"].quantile(0.95)),
        "median_summary_words": float(frame["summary_words"].median()),
        "mean_compression_ratio": float(frame["compression_ratio_words"].mean()),
        "over_4k_approx_pct": float(100 * (frame["combined_approx_tokens"] > 4096).mean()),
    })

distribution_report = pd.DataFrame(distribution_rows).round(3)
display(distribution_report)
print("Long records are intentionally retained here; notebook 03 will apply structure-aware processing.")

,partition,pairs,unique_cases,median_judgment_words,p95_judgment_words,median_summary_words,mean_compression_ratio,over_4k_approx_pct
0,train,6303,6303,3128.0,11271.90,641.0,0.235,62.208
1,validation,700,700,3296.0,11396.85,664.0,0.232,66.286
2,test_in_abs,100,100,3180.0,13920.10,596.5,0.221,62.000
3,test_in_ext_expert,100,50,4760.0,10693.00,1438.5,0.308,100.000
4,test_uk_abs,99,99,8882.0,33737.70,1072.0,0.129,98.990


Long records are intentionally retained here; notebook 03 will apply structure-aware processing.


## 10. Export case and pair manifests

These files contain ZIP member references and statistics, not duplicated full legal text. Notebook 03 will read the referenced members and build model-ready examples.

In [11]:
def save_manifest(frame, stem):
    csv_path = SPLIT_DIR / f"{stem}.csv"
    jsonl_path = SPLIT_DIR / f"{stem}.jsonl"
    frame.to_csv(csv_path, index=False)
    frame.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
    return csv_path, jsonl_path

exports = {}
for stem, frame in {
    "train_manifest": active_train_pairs,
    "validation_manifest": active_validation_pairs,
    "india_train_manifest": india_train_pairs,
    "india_validation_manifest": india_validation_pairs,
    "uk_train_manifest": uk_train_pairs,
    "uk_validation_manifest": uk_validation_pairs,
    "test_in_abs_manifest": in_abs_test_pairs,
    "test_in_ext_expert_manifest": in_ext_expert_pairs,
    "test_uk_abs_manifest": uk_abs_test_pairs,
    "train_cases": active_train_cases,
    "validation_cases": active_validation_cases,
}.items():
    exports[stem] = [str(path) for path in save_manifest(frame, stem)]

decisions_df = pd.DataFrame(decision_rows)
decisions_df.to_csv(VALIDATION_REPORT_DIR / "data_quality_decisions.csv", index=False)
cleaning_summary.to_csv(VALIDATION_REPORT_DIR / "cleaning_summary.csv", index=False)
final_leakage_report.to_csv(VALIDATION_REPORT_DIR / "final_leakage_report.csv", index=False)
distribution_report.to_csv(VALIDATION_REPORT_DIR / "split_distribution_report.csv", index=False)

split_registry = {
    "training_scope": TRAINING_SCOPE,
    "random_seed": RANDOM_SEED,
    "validation_fraction": VALIDATION_FRACTION,
    "active_train_cases": int(len(active_train_cases)),
    "active_validation_cases": int(len(active_validation_cases)),
    "test_in_abs_cases": int(len(in_abs_test_cases)),
    "test_in_ext_cases": int(len(in_ext_expert_cases)),
    "test_in_ext_reference_pairs": int(len(in_ext_expert_pairs)),
    "test_uk_abs_cases": int(len(uk_abs_test_cases)),
    "removed_for_evaluation_leakage": int(len(leakage_rows)),
    "removed_as_training_duplicates": int(len(duplicate_training_rows)),
    "exports": exports,
}
with open(VALIDATION_REPORT_DIR / "split_registry.json", "w", encoding="utf-8") as f:
    json.dump(split_registry, f, indent=2)

print(f"Split manifests saved to: {SPLIT_DIR}")
print(f"Validation reports saved to: {VALIDATION_REPORT_DIR}")
for path in sorted(SPLIT_DIR.iterdir()):
    print(" -", path.name)

Split manifests saved to: /data2/user_data/sg57092c/LLM_finetune/data/splits
Validation reports saved to: /data2/user_data/sg57092c/LLM_finetune/artifacts/data_validation
 - india_train_manifest.csv
 - india_train_manifest.jsonl
 - india_validation_manifest.csv
 - india_validation_manifest.jsonl
 - test_in_abs_manifest.csv
 - test_in_abs_manifest.jsonl
 - test_in_ext_expert_manifest.csv
 - test_in_ext_expert_manifest.jsonl
 - test_uk_abs_manifest.csv
 - test_uk_abs_manifest.jsonl
 - train_cases.csv
 - train_cases.jsonl
 - train_manifest.csv
 - train_manifest.jsonl
 - uk_train_manifest.csv
 - uk_train_manifest.jsonl
 - uk_validation_manifest.csv
 - uk_validation_manifest.jsonl
 - validation_cases.csv
 - validation_cases.jsonl
 - validation_manifest.csv
 - validation_manifest.jsonl


## 11. Handoff to notebook 03

`03_long_document_preprocessing.ipynb` should now:

1. Read `train_manifest.jsonl`, `validation_manifest.jsonl`, and each test manifest.
2. Load the referenced judgment and summary text directly from the original ZIP.
3. Calculate exact Llama token lengths for both inputs and targets.
4. Apply structure-aware chunking or extract-then-abstract preprocessing without splitting a case across partitions.
5. Align partial summaries with document sections where chunk-level supervision is used.
6. Produce final SFT examples and preserve `case_id`, `judgment_hash`, jurisdiction, source dataset, and source member paths for traceability.